In [7]:
import os
import re
from tqdm import tqdm
from bs4 import BeautifulSoup
import time
import uuid
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import cloudscraper
from urllib.parse import quote_plus
from fake_useragent import UserAgent

# Initialize fake user agent generator
ua = UserAgent()

# Generate headers with a random User-Agent
headers = {
    "User-Agent": ua.random,
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}


scraper = cloudscraper.create_scraper()





def clean_title(title):
    """
    Remove parentheses and everything inside from the title.
    Example: 'The Diary of a Young Girl (Mass Market Paperback)' -> 'The Diary of a Young Girl'
    """
    return re.sub(r"\s*\(.*?\)", "", title).strip()

    
def downld_epub_fast(epub_link, scraper, download_dir="download_dir", chunk_size=65536, max_workers=4):
    """
    Fast EPUB downloader with multiple optimizations:
    - Larger chunk size (64KB default)
    - Parallel chunk downloading for large files
    - Reduced system calls
    - Optimized file I/O
    """
    try:
        os.makedirs(download_dir, exist_ok=True)
        
        # First, get file info with HEAD request (faster than GET for metadata)
        head_response = scraper.head(epub_link, timeout=30)
        head_response.raise_for_status()
        
        # Extract filename from Content-Disposition
        cd = head_response.headers.get("content-disposition", "")
        match = re.search(r'filename="?([^"]+)"?', cd)
        if match:
            raw_name = match.group(1)
            final_filename = os.path.basename(raw_name.strip('"'))
        else:
            print("❌ No valid filename in headers")
            return None
            
        save_path = os.path.join(download_dir, final_filename)
        
        # Skip if already exists
        if os.path.exists(save_path):
            print(f"⏭️  File already exists: {save_path}")
            return save_path
            
        total_size = int(head_response.headers.get("content-length", 0))
        
        # For small files or when parallel download isn't beneficial, use simple download
        if total_size < 10 * 1024 * 1024:  # Less than 10MB
            return _simple_fast_download(epub_link, scraper, save_path, final_filename, total_size, chunk_size)
        
            
    except Exception as e:
        print(f"❌ Download failed: {e}")
        return None

def _simple_fast_download(epub_link, scraper, save_path, filename, total_size, chunk_size):
    """Optimized single-threaded download for smaller files"""
    with scraper.get(epub_link, stream=True, timeout=30) as response:
        response.raise_for_status()
        
        with open(save_path, "wb") as file, tqdm(
            desc=filename,
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            # Write chunks in larger batches to reduce system calls
            buffer = bytearray()
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    buffer.extend(chunk)
                    # Write buffer when it gets large enough
                    if len(buffer) >= chunk_size * 4:  # Write every ~256KB
                        file.write(buffer)
                        bar.update(len(buffer))
                        buffer.clear()
            
            # Write remaining buffer
            if buffer:
                file.write(buffer)
                bar.update(len(buffer))
    
    print(f"✅ Download complete: {save_path}")
    return save_path


def fetch_and_download(payload, scraper=None, download_dir="download_dir"):
    """
    Given a payload {id, filename}, handle the whole process:
      - POST to Fetching_Resource.php
      - Extract redirect link
      - Validate headers
      - Download if valid
    Returns the saved file path or None.
    """
    import re

    base_url = "https://oceanofpdf.com/Fetching_Resource.php"

    # Use provided scraper or create a new one
    scraper = scraper or cloudscraper.create_scraper()

    print(f"\n[+] Requesting resource for {payload['filename']}...")
    try:
        # Step 1: submit the form
        response = scraper.post(base_url, data=payload, timeout=20)
        response.raise_for_status()
        #print("Status:", response.status_code)
    except Exception as e:
        print(f"❌ POST request failed: {e}")
        return None

    # Step 2: look for redirect link
    match = re.search(r'https://fs\d+\.oceanofpdf\.com/[^\s"\']+', response.text)
    if not match:
        print("[!] No redirect URL found. Response preview:")
        print(response.text[:500])
        return None

    redirect_url = match.group(0)
    #print("[+] Found redirect:", redirect_url)

    # Step 3: HEAD request for headers only
    try:
        head_resp = scraper.head(redirect_url, allow_redirects=True, timeout=15)
        head_resp.raise_for_status()
    except Exception as e:
        print(f"❌ HEAD request failed: {e}")
        return None

    #print("\n[+] Redirect URL Headers:")
    #for k, v in head_resp.headers.items():
        #print(f"{k}: {v}")

    # Step 4: validate content-disposition
    cd = head_resp.headers.get("content-disposition", "")
    if "attachment" in cd and "filename=" in cd:
        #print("\n[+] Valid attachment found, starting download...")
        return downld_epub_fast(redirect_url, scraper, download_dir=download_dir,chunk_size=65536, max_workers=4)
        #return downld_epub(redirect_url, scraper, download_dir=download_dir)
    else:
        print("❌ No valid downloadable attachment in headers.")
        return None


def get_download_forms(book_url, scraper):
    """
    Fetch all download form details (id, filename) from a book page.
    Returns only EPUB forms if available, otherwise returns other formats.
    Returns a list of dicts like:
        [{"id": "srv3", "filename": "Book.epub"}, {"id": "srv4", "filename": "Book.pdf"}]
    """
    try:
        response = scraper.get(book_url,headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch {book_url}: {e}")
        return []
    
    soup = BeautifulSoup(response.text, "html.parser")
    forms = soup.find_all("form", action="https://oceanofpdf.com/Fetching_Resource.php")
    
    epub_forms = []
    other_forms = []
    
    for form in forms:
        id_input = form.find("input", {"name": "id"})
        filename_input = form.find("input", {"name": "filename"})
        
        if id_input and filename_input:
            file_ext = filename_input["value"].split(".")[-1].lower()
            form_data = {
                "id": id_input["value"],
                "filename": filename_input["value"]
            }
            
            if file_ext == "epub":
                epub_forms.append(form_data)
            else:
                other_forms.append(form_data)
    
    # Return only EPUB forms if available, otherwise return other formats
    if epub_forms:
        time.sleep(3)  # throttle requests
        return epub_forms
    else:
        time.sleep(3)  # throttle requests
        return other_forms


def get_last_page(url):
    """Find the last page number from pagination."""
    try:
        print(f"getting last page for {url}")
        response = scraper.get(url,headers=headers, timeout=15)
        response.raise_for_status()
    except Exception as e:
        print(f"[!] Failed to fetch {url}: {e}")
        return 1  # fallback: only page 1

    soup = BeautifulSoup(response.text, "html.parser")
    pagination_div = soup.find("div", class_="archive-pagination pagination")

    if not pagination_div:
        return 1

    page_numbers = []
    for a_tag in pagination_div.find_all("a", href=True):
        # remove <span> tags
        for span in a_tag.find_all("span"):
            span.decompose()

        text = a_tag.get_text(strip=True)
        if text.isdigit():
            page_numbers.append(int(text))

    time.sleep(3)

    return max(page_numbers) if page_numbers else 1




def Download_books_from_Oceanofpdf_Category(base_url, start_page=None, stop_page=None, max_pages=None, First_N_books=None):
    """
    Fetch all book article links across pagination pages.
    Supports optional start_page, stop_page, max_pages, and First_N_books limits.
    """
    category_name = None
    
    if base_url.startswith('https://oceanofpdf.com/category/genres/'):
        category_name = base_url.replace('https://oceanofpdf.com/category/genres/','').strip()

    # Detect last page if stop_page or max_pages not provided
    last_page = get_last_page(base_url)
    print(f"Detected last page: {last_page}")

    # Set defaults
    start_page = start_page or 1
    stop_page = stop_page or last_page

    # Apply max_pages if supplied
    if max_pages:
        stop_page = min(start_page + max_pages - 1, stop_page)

    print(f"Fetching from page {start_page} to {stop_page}")

    downloaded_count = 0  # Running count of downloaded books
    all_links = []

    for page in range(start_page, stop_page + 1):
        page_url = f"{base_url}page/{page}/"
        print(f"[+] Fetching page {page}: {page_url}")

        try:
            response = scraper.get(page_url, timeout=15)
            response.raise_for_status()
        except Exception as e:
            print(f"[!] Failed to fetch {page_url}: {e}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        # If category_name not set, fetch from page
        if category_name is None:
            h1_tag = soup.select_one(
                "div.archive-description.taxonomy-archive-description.taxonomy-description > h1.archive-title"
            )
            if h1_tag:
                category_name = h1_tag.text.strip().replace(" ", "-")
                print(f"[+] Detected category name: {category_name}")

        articles = soup.find_all("article")

        for article in articles:
            if First_N_books and downloaded_count >= First_N_books:
                print(f"\n✅ Reached limit of {First_N_books} books. Stopping.")
                return all_links

            postmetainfo = article.find("div", class_="postmetainfo")
            if postmetainfo:
                language_strong = postmetainfo.find("strong", string="Language: ")
                if language_strong:
                    language_text = language_strong.next_sibling
                    if language_text.strip().lower() != "english":
                        print(f"❌ Skipping non-English book")
                        continue  # Skip non-English books
            

            header = article.find("header", class_="entry-header")
            if header:
                a_tag = header.find("a", class_="entry-title-link", href=True)
                if a_tag:
                    book_url = a_tag["href"]
                    all_links.append(book_url)
                    payload_list = get_download_forms(book_url, scraper)
                    if not payload_list:
                        print("❌ No forms found on page.")
                        continue

                    if isinstance(payload_list, dict):
                        path = fetch_and_download(payload_list, scraper, download_dir=f"download_dir_{category_name}")
                        if path:
                            downloaded_count += 1
                            print(f"📥 Downloaded books: {downloaded_count}")
                    elif isinstance(payload_list, list):
                        for payload in payload_list:
                            path = fetch_and_download(payload, scraper, download_dir=f"download_dir_{category_name}")
                            if path:
                                downloaded_count += 1
                                print(f"📥 Downloaded books: {downloaded_count}")

        time.sleep(5)

    print(f"\nTotal Book Links Collected: {len(all_links)}")
    return all_links



In [9]:
if __name__ == "__main__":
    harlequin_urls = [
        "https://oceanofpdf.com/category/genres/harlequin-desire/",
        "https://oceanofpdf.com/category/genres/harlequin-historical/",
        "https://oceanofpdf.com/category/genres/harlequin-medical-romance/",
        "https://oceanofpdf.com/category/genres/harlequin-nocturne/",
        "https://oceanofpdf.com/category/genres/harlequin-romantic-suspense/"
    ]

    all_book_links = []

    for url in harlequin_urls:
        print(f"\nFetching books from: {url}")
        book_links = Download_books_from_Oceanofpdf_Category(base_url=url)

    print("\n=== All Book Links Collected ===")




Fetching books from: https://oceanofpdf.com/category/genres/harlequin-desire/
getting last page for https://oceanofpdf.com/category/genres/harlequin-desire/
Detected last page: 71
Fetching from page 1 to 71
[+] Fetching page 1: https://oceanofpdf.com/category/genres/harlequin-desire/page/1/

[+] Requesting resource for Lessons_in_seduction_-_Sandra_Hyatt.epub...
⏭️  File already exists: download_dir_harlequin-desire/Lessons_in_seduction_-_Sandra_Hyatt.epub
📥 Downloaded books: 1

[+] Requesting resource for Texas_millionaire_-_Dixie_browning.epub...
⏭️  File already exists: download_dir_harlequin-desire/Texas_millionaire_-_Dixie_browning.epub
📥 Downloaded books: 2

[+] Requesting resource for Taming_the_texas_tycoon_-_Katherine_garbera.epub...
⏭️  File already exists: download_dir_harlequin-desire/Taming_the_texas_tycoon_-_Katherine_garbera.epub
📥 Downloaded books: 3

[+] Requesting resource for In_too_deep_-_Brenda_Jackson.epub...
⏭️  File already exists: download_dir_harlequin-desire

KeyboardInterrupt: 